1. [Условие](https://drive.google.com/file/d/1p8VqZtq48yBSReqCjptTsCong8g4Y2_e/view)
2. [Исходные данные](https://drive.google.com/drive/folders/1PDlb4Zyg_mG5utzY3b38kzKQWAvPzAPm)

В июле 2026 года был проведен эксперимент по внедрению **новой AI-рекомендательной модели на главной странице**. Новая модель предназначена для подбора более персонализированного набора рекомендаций на основе предпочтения пользователей

**Срок эксперимента: 30 дней**

# **1. Проверка качества данных**

In [1]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
cfg = {
    'clicks' : 'clicks.csv',
    'data_dictionary_path' : 'data_dictionary.csv',
    'exp_assignment' : 'experiment_assignments.csv',
    'impressions' : 'impressions.csv',
    'orders' : 'orders.csv',
    'premium' : 'premium_subscriptions.csv',
    'products' : 'products.csv',
    'sellers' : 'sellers.csv',
    'sessions' : 'sessions.csv',
    'support_tickets' : 'support_tickets.csv',
    'users' : 'users.csv'
}

for name, value in cfg.items():
        cfg[name] = os.path.join('data', value)

Последовательно проверим все данные на целостность, полноту, пропуски, дубли и выбросы

## **Таблица Пользователей**

In [3]:
users = pd.read_csv(cfg['users'])
users.registration_date = users.registration_date.astype('datetime64[s]')
display(users.sample(5))
users.info()
users.describe(include='all')

,user_id,registration_date,region,device,age_group,traffic_source,is_premium,is_bot_candidate
42232,u_042233,2026-06-04,Siberia,ios,18-24,direct,0,0
18489,u_018490,2025-03-17,Central Russia,web,45-54,paid_ads,0,0
41879,u_041880,2026-05-24,Central Russia,ios,25-34,push,0,0
44051,u_044052,2024-12-05,Central Russia,android,35-44,push,0,0
22846,u_022847,2026-06-26,Volga,ios,18-24,organic,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   user_id            50000 non-null  object       
 1   registration_date  50000 non-null  datetime64[s]
 2   region             50000 non-null  object       
 3   device             50000 non-null  object       
 4   age_group          50000 non-null  object       
 5   traffic_source     50000 non-null  object       
 6   is_premium         50000 non-null  int64        
 7   is_bot_candidate   50000 non-null  int64        
dtypes: datetime64[s](1), int64(2), object(5)
memory usage: 3.1+ MB


,user_id,registration_date,region,device,age_group,traffic_source,is_premium,is_bot_candidate
count,50000,50000,50000,50000,50000,50000,50000.000000,50000.000000
unique,50000,NaN,8,3,5,6,NaN,NaN
top,u_000001,NaN,Central Russia,android,25-34,organic,NaN,NaN
freq,1,NaN,10957,28894,16754,21129,NaN,NaN
mean,NaN,2025-11-04 00:08:05,NaN,NaN,NaN,NaN,0.165080,0.008000
min,NaN,2017-05-18 00:00:00,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,NaN,2025-08-05 00:00:00,NaN,NaN,NaN,NaN,0.000000,0.000000
50%,NaN,2026-01-17 00:00:00,NaN,NaN,NaN,NaN,0.000000,0.000000
75%,NaN,2026-04-23 00:00:00,NaN,NaN,NaN,NaN,0.000000,0.000000
max,NaN,2026-07-01 00:00:00,NaN,NaN,NaN,NaN,1.000000,1.000000


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **user_id** | object | Уникальный идентификатор пользователя. Primary key таблицы. |
| **registration_date** | date | Дата регистрации пользователя в сервисе. |
| **region** | object | Макрорегион пользователя. В данных представлено 8 регионов. |
| **device** | object | Основной тип устройства пользователя. Возможные значения: `android`, `ios`, `web`. |
| **age_group** | object | Возрастная группа пользователя. В данных представлено 5 возрастных категорий. |
| **traffic_source** | object | Основной источник привлечения пользователя (канал привлечения). В таблице представлено 6 различных источников трафика. |
| **is_premium** | int64 | Наиличие Premium-подписки на момент начала эксперимента (`1` - Premium, `0` - нет). Доля Premium-пользователей составляет около **16.5%**. |
| **is_bot_candidate** | int64 | Синтетический флаг потенциальной автоматизированной активности (`1` - пользователь помечен как возможный бот, `0` - обычный пользователь). Доля подозрительных пользователей составляет около **0.8%**. |

## **Таблица Сплитования [Experiment_assignment]**

In [4]:
exp_assignment = pd.read_csv(cfg['exp_assignment'], index_col='assignment_id')
exp_assignment.assignment_date = exp_assignment.assignment_date.astype('datetime64[s]')

display(exp_assignment.sample(5))
exp_assignment.info()
exp_assignment.describe(include='all')

,user_id,experiment_name,experiment_group,assignment_date,assignment_source
assignment_id,,,,,
a_026083_1,u_026083,ai_recommendations_homepage,test,2026-07-01,assignment_service
a_038881_1,u_038881,ai_recommendations_homepage,control,2026-07-01,assignment_service
a_024865_1,u_024865,ai_recommendations_homepage,control,2026-07-01,assignment_service
a_015762_1,u_015762,ai_recommendations_homepage,test,2026-07-01,assignment_service
a_049621_1,u_049621,ai_recommendations_homepage,control,2026-07-01,assignment_service


<class 'pandas.core.frame.DataFrame'>
Index: 50300 entries, a_000001_1 to a_050000_1
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   user_id            50300 non-null  object       
 1   experiment_name    50300 non-null  object       
 2   experiment_group   50300 non-null  object       
 3   assignment_date    50300 non-null  datetime64[s]
 4   assignment_source  50300 non-null  object       
dtypes: datetime64[s](1), object(4)
memory usage: 2.3+ MB


,user_id,experiment_name,experiment_group,assignment_date,assignment_source
count,50300,50300,50300,50300,50300
unique,50000,1,2,NaN,2
top,u_011205,ai_recommendations_homepage,test,NaN,assignment_service
freq,2,50300,25326,NaN,50000
mean,NaN,NaN,NaN,2026-07-01 00:26:23,NaN
min,NaN,NaN,NaN,2026-07-01 00:00:00,NaN
25%,NaN,NaN,NaN,2026-07-01 00:00:00,NaN
50%,NaN,NaN,NaN,2026-07-01 00:00:00,NaN
75%,NaN,NaN,NaN,2026-07-01 00:00:00,NaN
max,NaN,NaN,NaN,2026-07-06 00:00:00,NaN


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **assignment_id** | object | Уникальный идентификатор события назначения пользователя в эксперимент. Primary key таблицы. |
| **user_id** | object | Уникальный идентификатор пользователя. Foreign key на таблицу `users`. Для части пользователей в таблице присутствуют повторные записи назначения. |
| **experiment_name** | object | Название эксперимента. В рассматриваемых данных присутствует один эксперимент - `ai_recommendations_homepage`. |
| **experiment_group** | object | Экспериментальная группа пользователя. Возможные значения: `control` и `test`. |
| **assignment_date** | date | Дата назначения пользователя в эксперимент. В исходном файле хранится как `object` и требует преобразования в тип `datetime`. |
| **assignment_source** | object | Источник назначения пользователя в эксперимент. Используется для идентификации механизма рандомизации. В данных представлено 2 уникальных значения. |

**!!!Таблица содержит 50 300 записей для 50 000 пользователей** - часть пользователей имеет повторные записи назначения в эксперимент, что соответствует описанию данных и требует проверки при подготовке выборки для A/B-теста.

In [5]:
# Удалим всех пользователей, для которых было допущено двойное присвоение для обеспечения чистоты эксперимента

dub_users = exp_assignment.groupby('user_id').filter(lambda x: x.shape[0] > 1)
exp_assignment = exp_assignment[~exp_assignment.user_id.isin(dub_users.user_id)]
exp_assignment.shape

(49700, 5)

## **Таблица cессий**

In [22]:
sessions = pd.read_csv(cfg['sessions'], index_col='session_id')
sessions.session_date = sessions.session_date.astype('datetime64[s]')
sessions = sessions.drop('experiment_group', axis=1)

display(sessions.sample(5))
sessions.info()
sessions.describe(include='all')

,user_id,session_date,device,region,session_duration_sec,page_views,entry_point
session_id,,,,,,,
ss_00050832,u_015430,2026-07-19,ios,Saint Petersburg,262,2,category
ss_00140392,u_042514,2026-07-07,ios,Volga,226,3,homepage
ss_00022678,u_006914,2026-07-16,android,Ural,158,7,product_page
ss_00058041,u_017585,2026-07-03,android,Moscow,81,6,homepage
ss_00094164,u_028545,2026-07-13,web,Saint Petersburg,249,11,homepage


<class 'pandas.core.frame.DataFrame'>
Index: 164739 entries, ss_00000001 to ss_00164739
Data columns (total 7 columns):
 #   Column                Non-Null Count   Dtype        
---  ------                --------------   -----        
 0   user_id               164739 non-null  object       
 1   session_date          164739 non-null  datetime64[s]
 2   device                164739 non-null  object       
 3   region                164739 non-null  object       
 4   session_duration_sec  164739 non-null  int64        
 5   page_views            164739 non-null  int64        
 6   entry_point           164739 non-null  object       
dtypes: datetime64[s](1), int64(2), object(4)
memory usage: 10.1+ MB


,user_id,session_date,device,region,session_duration_sec,page_views,entry_point
count,164739,164739,164739,164739,164739.000000,164739.000000,164739
unique,50000,NaN,3,8,NaN,NaN,5
top,u_033963,NaN,android,Central Russia,NaN,NaN,homepage
freq,32,NaN,94931,35854,NaN,NaN,56092
mean,NaN,2026-07-15 11:19:34,NaN,NaN,294.497284,8.126412,NaN
min,NaN,2026-07-01 00:00:00,NaN,NaN,12.000000,1.000000,NaN
25%,NaN,2026-07-08 00:00:00,NaN,NaN,146.000000,5.000000,NaN
50%,NaN,2026-07-15 00:00:00,NaN,NaN,228.000000,7.000000,NaN
75%,NaN,2026-07-23 00:00:00,NaN,NaN,360.000000,9.000000,NaN
max,NaN,2026-07-30 00:00:00,NaN,NaN,7200.000000,59.000000,NaN


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **session_id** | object | Уникальный идентификатор пользовательской сессии. Primary key таблицы. |
| **user_id** | object | Уникальный идентификатор пользователя. Foreign key на таблицу `users`. Один пользователь может иметь несколько сессий. |
| **session_date** | datetime | Дата начала пользовательской сессии. |
| **device** | object | Тип устройства, с которого была инициирована сессия. Возможные значения: `android`, `ios`, `web`. |
| **region** | object | Макрорегион пользователя. В данных представлено 8 регионов. |
| **experiment_group** | object | Экспериментальная группа, активная в рамках данной сессии. Возможные значения: `control` и `test`. Для пользователей с повторным назначением в эксперименте значение может отличаться между сессиями. |
| **session_duration_sec** | int64 | Длительность пользовательской сессии в секундах. Значения находятся в диапазоне от 12 до 7200 секунд (2 часа). |
| **page_views** | int64 | Количество просмотренных страниц в рамках сессии. Значения находятся в диапазоне от 1 до 59 страниц. |
| **entry_point** | object | Точка входа пользователя в сессию. В данных представлено 5 различных источников входа. |

**Особенность данных:** один пользователь может иметь несколько сессий. Для пользователей с повторным назначением в эксперименте (`duplicate assignment`) группа эксперимента может отличаться между сессиями, что необходимо учитывать при расчете продуктовых метрик и анализе A/B-теста.

## **Таблица показов**

In [7]:
impressions = pd.read_csv(cfg['impressions'], index_col='impression_id')
impressions.event_time = impressions.event_time.astype('datetime64[s]')

display(impressions.sample(5))
impressions.info()
impressions.describe(include='all')


,session_id,user_id,event_time,product_id,position,model_version,is_clicked,is_duplicate_event
impression_id,,,,,,,,
imp_0000957923,ss_00148053,u_044864,2026-07-29 12:29:57,p_01645,2,ai_v2,0,0
imp_0000102097,ss_00015764,u_004845,2026-07-01 21:42:04,p_03010,2,baseline_v1,0,0
imp_0000454027,ss_00070086,u_021227,2026-07-27 12:19:33,p_03412,2,ai_v2,0,0
imp_0000462435,ss_00071396,u_021617,2026-07-25 13:36:36,p_03401,4,ai_v2,0,0
imp_0000553278,ss_00085437,u_025857,2026-07-08 18:51:16,p_01207,11,baseline_v1,0,0


<class 'pandas.core.frame.DataFrame'>
Index: 1066434 entries, imp_0000000001 to imp_0001066434
Data columns (total 8 columns):
 #   Column              Non-Null Count    Dtype        
---  ------              --------------    -----        
 0   session_id          1066434 non-null  object       
 1   user_id             1066434 non-null  object       
 2   event_time          1066434 non-null  datetime64[s]
 3   product_id          1066434 non-null  object       
 4   position            1066434 non-null  int64        
 5   model_version       1066434 non-null  object       
 6   is_clicked          1066434 non-null  int64        
 7   is_duplicate_event  1066434 non-null  int64        
dtypes: datetime64[s](1), int64(3), object(4)
memory usage: 73.2+ MB


,session_id,user_id,event_time,product_id,position,model_version,is_clicked,is_duplicate_event
count,1066434,1066434,1066434,1066434,1.066434e+06,1066434,1.066434e+06,1.066434e+06
unique,164739,50000,NaN,4500,NaN,2,NaN,NaN
top,ss_00129691,u_033963,NaN,p_02210,NaN,ai_v2,NaN,NaN
freq,12,223,NaN,299,NaN,536188,NaN,NaN
mean,NaN,NaN,2026-07-16 02:42:41,NaN,4.143905e+00,NaN,8.558054e-02,2.994091e-03
min,NaN,NaN,2026-07-01 08:00:00,NaN,1.000000e+00,NaN,0.000000e+00,0.000000e+00
25%,NaN,NaN,2026-07-08 14:45:56,NaN,2.000000e+00,NaN,0.000000e+00,0.000000e+00
50%,NaN,NaN,2026-07-15 22:07:36,NaN,4.000000e+00,NaN,0.000000e+00,0.000000e+00
75%,NaN,NaN,2026-07-23 15:08:44,NaN,6.000000e+00,NaN,0.000000e+00,0.000000e+00
max,NaN,NaN,2026-07-30 22:59:56,NaN,1.200000e+01,NaN,1.000000e+00,1.000000e+00


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **impression_id** | object | Уникальный идентификатор события показа рекомендации. Primary key таблицы. |
| **session_id** | object | Идентификатор пользовательской сессии. Foreign key на таблицу `sessions`. Используется для связи показов рекомендаций с конкретным визитом пользователя. |
| **user_id** | object | Идентификатор пользователя. Foreign key на таблицу `users`. Один пользователь может иметь множество показов рекомендаций. |
| **event_time** | datetime | Время показа рекомендованного товара пользователю. |
| **product_id** | object | Идентификатор рекомендованного товара. В данных представлено 4500 уникальных товаров. |
| **position** | int | Позиция товара в рекомендательной выдаче. Значения находятся в диапазоне от 1 до 12. |
| **model_version** | object | Версия рекомендательной модели, которая сформировала показ. В данных представлены две версии: `baseline_v1` и `ai_v2`. |
| **is_clicked** | int64 | Бинарный признак взаимодействия пользователя с рекомендацией (`1` - по показу был совершен клик, `0` - клик не совершен).|
| **is_duplicate_event** | int64 | Флаг потенциального дублирования события (`1` - событие отмечено как дубликат, `0` - уникальное событие). Доля дубликатов составляет около **0.3%**. |

**Особенность эксперимента:** в данных представлены две версии рекомендательной модели:
- `baseline_v1` - контрольная версия модели;
- `ai_v2` - новая AI-рекомендательная модель.

Таблица является основной для оценки эффективности рекомендательной системы и расчета метрик взаимодействия пользователей с рекомендациями (CTR, position bias, conversion rate).

In [8]:
# Удалим дублирующиеся события

impressions = impressions[impressions.is_duplicate_event == 0]
impressions.shape

(1063241, 8)

## **Таблица кликов**

In [9]:
clicks = pd.read_csv(cfg['clicks'], index_col='click_id')
clicks.event_time = clicks.event_time.astype('datetime64[s]')
display(clicks.sample(5))
clicks.info()
clicks.describe(include='all')

,impression_id,session_id,user_id,event_time,product_id,position
click_id,,,,,,
clk_0000063967,imp_0000749706,ss_00115840,u_035085,2026-07-03 21:37:49,p_03504,2
clk_0000085632,imp_0000999717,ss_00154471,u_046803,2026-07-13 20:38:08,p_02109,2
clk_0000076581,imp_0000895578,ss_00138406,u_041950,2026-07-30 18:05:42,p_01038,1
clk_0000020654,imp_0000244802,ss_00037756,u_011469,2026-07-13 11:14:22,p_03376,3
clk_0000071407,imp_0000835205,ss_00129006,u_039068,2026-07-09 20:09:04,p_04469,1


<class 'pandas.core.frame.DataFrame'>
Index: 91266 entries, clk_0000000001 to clk_0000091266
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype        
---  ------         --------------  -----        
 0   impression_id  91266 non-null  object       
 1   session_id     91266 non-null  object       
 2   user_id        91266 non-null  object       
 3   event_time     91266 non-null  datetime64[s]
 4   product_id     91266 non-null  object       
 5   position       91266 non-null  int64        
dtypes: datetime64[s](1), int64(1), object(4)
memory usage: 4.9+ MB


,impression_id,session_id,user_id,event_time,product_id,position
count,91266,91266,91266,91266,91266,91266.000000
unique,91266,71373,37397,NaN,4500,NaN
top,imp_0000000007,ss_00036682,u_049001,NaN,p_02872,NaN
freq,1,7,41,NaN,38,NaN
mean,NaN,NaN,NaN,2026-07-16 03:15:10,NaN,3.219425
min,NaN,NaN,NaN,2026-07-01 08:00:02,NaN,1.000000
25%,NaN,NaN,NaN,2026-07-08 15:02:18,NaN,2.000000
50%,NaN,NaN,NaN,2026-07-15 22:41:40,NaN,3.000000
75%,NaN,NaN,NaN,2026-07-23 15:21:04,NaN,4.000000
max,NaN,NaN,NaN,2026-07-30 22:59:55,NaN,12.000000


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **click_id** | object | Идентификатор клика. Primary key |
| **impression_id** | object | Уникальный идентификатор события показа по которому был произведен клик. Первичный ключ таблицы. |
| **session_id** | object | Идентификатор пользовательской сессии. Используется для объединения событий в рамках одного визита. |
| **user_id** | object | Уникальный идентификатор пользователя. Один пользователь может иметь несколько сессий и событий. |
| **event_time** | datetime | Время совершения события. |
| **product_id** | object | Идентификатор товара, по которому произошло событие. В данных встречается 4500 уникальных товаров. |
| **position** | int64 | Позиция товара в рекомендательной выдаче на главной странице. Значения находятся в диапазоне от 1 до 12. Используется для анализа влияния позиции на вероятность клика (position bias). |

## **Таблица заказов**

In [21]:
orders = pd.read_csv(cfg['orders'], index_col='order_id')
orders.order_date = orders.order_date.astype('datetime64[s]')
orders.return_date = orders.return_date.astype('datetime64[s]')
orders = orders.drop('category', axis=1)

display(orders.head(5))
orders.info()
orders.describe(include='all')

,user_id,session_id,click_id,order_date,product_id,gmv_rub,discount_rub,delivery_fee_rub,payment_type,is_returned,return_date,gross_margin_rub
order_id,,,,,,,,,,,,
ord_000000001,u_000007,ss_00000016,clk_0000000007,2026-07-02,p_01555,4374,0,83.0,bank_card,0,NaT,879.17
ord_000000002,u_000020,ss_00000070,clk_0000000059,2026-07-14,p_03428,2690,83,120.0,sberpay,0,NaT,407.27
ord_000000003,u_000024,ss_00000081,clk_0000000064,2026-07-02,p_04144,189,10,264.0,bank_card,0,NaT,28.92
ord_000000004,u_000028,ss_00000097,clk_0000000068,2026-07-15,p_02740,614,19,208.0,bank_card,0,NaT,127.83
ord_000000005,u_000037,ss_00000131,clk_0000000079,2026-07-10,p_02358,363,40,182.0,bank_card,0,NaT,50.31


<class 'pandas.core.frame.DataFrame'>
Index: 7407 entries, ord_000000001 to ord_000007407
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype        
---  ------            --------------  -----        
 0   user_id           7407 non-null   object       
 1   session_id        7407 non-null   object       
 2   click_id          7407 non-null   object       
 3   order_date        7407 non-null   datetime64[s]
 4   product_id        7407 non-null   object       
 5   gmv_rub           7407 non-null   int64        
 6   discount_rub      7407 non-null   int64        
 7   delivery_fee_rub  7104 non-null   float64      
 8   payment_type      7407 non-null   object       
 9   is_returned       7407 non-null   int64        
 10  return_date       462 non-null    datetime64[s]
 11  gross_margin_rub  7407 non-null   float64      
dtypes: datetime64[s](2), float64(2), int64(3), object(5)
memory usage: 752.3+ KB


,user_id,session_id,click_id,order_date,product_id,gmv_rub,discount_rub,delivery_fee_rub,payment_type,is_returned,return_date,gross_margin_rub
count,7407,7407,7407,7407,7407,7407.000000,7407.000000,7104.000000,7407,7407.000000,462,7407.000000
unique,6734,7265,7407,NaN,3644,NaN,NaN,NaN,5,NaN,NaN,NaN
top,u_021059,ss_00092480,clk_0000000007,NaN,p_01821,NaN,NaN,NaN,bank_card,NaN,NaN,NaN
freq,4,3,1,NaN,8,NaN,NaN,NaN,3283,NaN,NaN,NaN
mean,NaN,NaN,NaN,2026-07-15 07:58:50,NaN,1676.464156,67.189010,150.456926,NaN,0.062373,2026-08-05 05:49:05,274.497984
min,NaN,NaN,NaN,2026-07-01 00:00:00,NaN,84.000000,0.000000,0.000000,NaN,0.000000,2026-07-05 00:00:00,-331.980000
25%,NaN,NaN,NaN,2026-07-08 00:00:00,NaN,615.000000,0.000000,16.500000,NaN,0.000000,2026-07-26 00:00:00,103.965000
50%,NaN,NaN,NaN,2026-07-15 00:00:00,NaN,1165.000000,24.000000,166.000000,NaN,0.000000,2026-08-05 00:00:00,213.770000
75%,NaN,NaN,NaN,2026-07-23 00:00:00,NaN,2140.500000,79.000000,233.000000,NaN,0.000000,2026-08-16 00:00:00,382.995000
max,NaN,NaN,NaN,2026-07-30 00:00:00,NaN,21926.000000,2193.000000,481.000000,NaN,1.000000,2026-09-04 00:00:00,2433.040000


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **order_id** | object | Уникальный идентификатор заказа. Primary key таблицы. |
| **user_id** | object | Идентификатор пользователя, совершившего заказ. |
| **session_id** | object | Идентификатор пользовательской сессии, в рамках которой был оформлен заказ.|
| **click_id** | object | Идентификатор клика, после которого был совершен заказ. Foreign key на таблицу `clicks`. Используется для анализа конверсии рекомендаций в покупки. |
| **order_date** | datetime | Дата оформления заказа. |
| **product_id** | object | Идентификатор приобретенного товара. Foreign key на таблицу `products`. В данных представлено 3644 уникальных товара. |
| **category** | object | Категория приобретенного товара. В данных представлено 10 категорий товаров. |
| **gmv_rub** | int64 | Gross Merchandise Value — стоимость заказа в рублях после учета скидок, но до учета возвратов. Значения находятся в диапазоне от 84 до 21 926 рублей. |
| **discount_rub** | int64 | Размер скидки в рублях, предоставленной пользователю при оформлении заказа. **Средний размер скидки составляет около 67 рублей.** |
| **delivery_fee_rub** | float64 | Стоимость доставки в рублях. **Содержит пропущенные значения.** |
| **payment_type** | object | Тип оплаты заказа. В данных представлено 5 вариантов оплаты. |
| **is_returned** | int64 | Бинарный признак возврата заказа (`1` — заказ возвращен, `0` — нет). Доля возвратов составляет около **6.2%**. |
| **return_date** | datetime | Дата возврата заказа. Заполнена только для возвращенных заказов (`462` записи). Для остальных заказов содержит `NaN`. |
| **gross_margin_rub** | float64 | Оценка валовой прибыли с заказа в рублях. Для возвращенных заказов может принимать небольшие отрицательные значения. Среднее значение составляет около 274.5 рублей. |

**Особенности данных:**
- Таблица является основной для расчета бизнес-метрик эксперимента:
  - **Conversion Rate** — доля кликов, завершившихся покупкой;
  - **GMV uplift** — изменение объема продаж между группами;
  - **Gross Profit uplift** — изменение валовой прибыли;
  - **Return Rate** — доля возвратов.
- Наличие отрицательных значений `gross_margin_rub` для возвращенных заказов позволяет учитывать влияние возвратов на реальную прибыльность рекомендаций.

## **Таблица товаров**

In [11]:
products = pd.read_csv(cfg['products'], index_col='product_id')
products.created_date = products.created_date.astype('datetime64[s]')
display(products.sample(5))
products.info()
products.describe(include='all')

,category,seller_id,base_price_rub,gross_margin_rate,product_rating,is_private_label,created_date
product_id,,,,,,,
p_03400,Fashion,s_0589,1569,0.2889,4.32,0,2021-10-07
p_02037,Sports,s_0414,908,0.1984,4.57,0,2022-11-19
p_00480,Beauty,s_0421,403,0.3685,4.24,0,2021-01-08
p_00653,Electronics,s_0098,2625,0.0400,4.89,0,2024-02-03
p_00539,Home,s_0365,1611,0.1935,4.30,0,2023-06-25


<class 'pandas.core.frame.DataFrame'>
Index: 4500 entries, p_00001 to p_04500
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   category           4500 non-null   object       
 1   seller_id          4500 non-null   object       
 2   base_price_rub     4500 non-null   int64        
 3   gross_margin_rate  4500 non-null   float64      
 4   product_rating     4500 non-null   float64      
 5   is_private_label   4500 non-null   int64        
 6   created_date       4500 non-null   datetime64[s]
dtypes: datetime64[s](1), float64(2), int64(2), object(2)
memory usage: 281.2+ KB


,category,seller_id,base_price_rub,gross_margin_rate,product_rating,is_private_label,created_date
count,4500,4500,4500.000000,4500.000000,4500.000000,4500.000000,4500
unique,10,700,NaN,NaN,NaN,NaN,NaN
top,Fashion,s_0661,NaN,NaN,NaN,NaN,NaN
freq,719,16,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,1717.671778,0.197618,4.439000,0.077556,2023-08-06 11:53:55
min,NaN,NaN,99.000000,0.040000,3.120000,0.000000,2021-01-01 00:00:00
25%,NaN,NaN,637.000000,0.151200,4.210000,0.000000,2022-04-26 00:00:00
50%,NaN,NaN,1199.000000,0.197750,4.450000,0.000000,2023-08-09 00:00:00
75%,NaN,NaN,2208.500000,0.242225,4.680000,0.000000,2024-11-15 00:00:00
max,NaN,NaN,21926.000000,0.427800,5.000000,1.000000,2026-03-14 00:00:00


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **product_id** | object | Уникальный идентификатор товара. Primary key таблицы. |
| **category** | object | Категория товара. В данных представлено 10 товарных категорий. |
| **seller_id** | object | Идентификатор продавца товара. Foreign key на таблицу продавцов. В каталоге представлено 700 уникальных продавцов. |
| **base_price_rub** | int64 | Базовая цена товара в рублях. |
| **gross_margin_rate** | float64 | Доля валовой маржи товара. Характеризует отношение прибыли маркетплейса от продажи к цене товара. |
| **product_rating** | float64 | Средняя оценка товара пользователями.|
| **is_private_label** | int64 | Бинарный признак собственного бренда (`1` — товар собственной торговой марки, `0` — сторонний товар). Доля private label товаров составляет около **7.8%**. |
| **created_date** | date | Дата создания товара в каталоге. |

**Особенности данных:**
- Таблица используется как справочник товаров для обогащения событий рекомендаций (`impressions`, `clicks`) дополнительными характеристиками:
  - ценой товара;
  - категорией;
  - маржинальностью;
  - качеством товара (rating);
  - принадлежностью к private label.
- Эти признаки могут использоваться при анализе влияния характеристик товара на эффективность AI-рекомендаций и бизнес-метрики (CTR, GMV, margin uplift).

## **Таблица обратной связи**

In [12]:
support_tickets = pd.read_csv(cfg['support_tickets'], index_col='ticket_id')
support_tickets.ticket_date = support_tickets.ticket_date.astype('datetime64[s]')
display(support_tickets.sample(5))
support_tickets.info()
support_tickets.describe(include='all')

,user_id,order_id,ticket_date,reason,sentiment_score,resolved_within_24h
ticket_id,,,,,,
t_00000259,u_038754,ord_000005772,2026-07-20,product_quality,0.062,1
t_00000017,u_002873,ord_000000428,2026-07-29,product_quality,-0.230,0
t_00000283,u_042796,ord_000006372,2026-07-30,delivery_delay,0.917,0
t_00000165,u_024685,ord_000003653,2026-07-30,product_quality,-0.304,1
t_00000151,u_023112,ord_000003415,2026-07-10,product_quality,-0.032,1


<class 'pandas.core.frame.DataFrame'>
Index: 337 entries, t_00000001 to t_00000337
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype        
---  ------               --------------  -----        
 0   user_id              337 non-null    object       
 1   order_id             337 non-null    object       
 2   ticket_date          337 non-null    datetime64[s]
 3   reason               337 non-null    object       
 4   sentiment_score      337 non-null    float64      
 5   resolved_within_24h  337 non-null    int64        
dtypes: datetime64[s](1), float64(1), int64(1), object(3)
memory usage: 18.4+ KB


,user_id,order_id,ticket_date,reason,sentiment_score,resolved_within_24h
count,337,337,337,337,337.000000,337.000000
unique,336,337,NaN,6,NaN,NaN
top,u_048300,ord_000000002,NaN,product_quality,NaN,NaN
freq,2,1,NaN,116,NaN,NaN
mean,NaN,NaN,2026-07-17 23:38:38,NaN,0.071896,0.715134
min,NaN,NaN,2026-07-02 00:00:00,NaN,-1.000000,0.000000
25%,NaN,NaN,2026-07-11 00:00:00,NaN,-0.294000,0.000000
50%,NaN,NaN,2026-07-17 00:00:00,NaN,0.080000,1.000000
75%,NaN,NaN,2026-07-26 00:00:00,NaN,0.414000,1.000000
max,NaN,NaN,2026-08-03 00:00:00,NaN,1.000000,1.000000


### Описание столбцов

| Столбец | Тип | Описание |
|---------|-----|----------|
| **ticket_id** | object | Уникальный идентификатор обращения в службу поддержки. Primary key таблицы. |
| **user_id** | object | Идентификатор пользователя, создавшего обращение. Foreign key на таблицу `users`. Один пользователь может иметь несколько обращений. |
| **order_id** | object | Идентификатор заказа, с которым связано обращение. Foreign key на таблицу `orders`. |
| **ticket_date** | datetime | Дата создания обращения в службу поддержки. |
| **reason** | object | Причина обращения пользователя. В данных представлено 6 категорий обращений. |
| **sentiment_score** | float64 | Синтетическая оценка тональности обращения в диапазоне от -1 до +1. Отрицательные значения соответствуют негативной обратной связи, положительные - позитивной. Среднее значение составляет 0.072. |
| **resolved_within_24h** | int64 | Бинарный признак скорости решения обращения (`1` - обращение закрыто в течение 24 часов, `0` - нет). Доля обращений, решенных в течение 24 часов, составляет около **71.5%**. |

**Особенности данных:**
- Таблица используется для анализа пользовательского опыта (Customer Experience, CX), включая:
  - связь качества рекомендаций с количеством обращений в поддержку;
  - анализ причин недовольства пользователей;
  - оценку влияния скорости решения проблем на удовлетворенность клиентов.
- Наиболее распространенная причина обращения - `product_quality` (116 обращений).

# **2. Загрузка данных в БД**

In [13]:
import sqlalchemy
from sqlalchemy import text

# Connecting to the Postgres Server
from db_conn import engine

with engine.begin() as conn:
    query = text("""
         
    """)